[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day01-llm-inference-stack.ipynb)

# Day 1 — The LLM inference stack: from prompt to tokens per second

Today you trace a prompt through the four stages of inference (tokenize → prefill → decode → detokenize), measure **tokens/sec** yourself, and compute the **KV-cache size** for GPT-2 with real numbers.

Runs on CPU or a single free T4. ~30 minutes. Run every cell top to bottom: **Runtime → Run all**.

## 0. Environment check

First, see what we're working with. `torch` may not be installed yet on a fresh runtime — that's fine, we install it next.

In [ ]:
import sys
print('python:', sys.version.split()[0])
try:
    import torch
    print('torch:', torch.__version__)
    print('cuda available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu:', torch.cuda.get_device_name(0))
except ImportError:
    print('torch not installed yet — installing in the next cell')

## 1. Install dependencies

Colab convention: `%pip` (restarts-safe, quiet). Skip if torch/transformers are already present.

In [ ]:
%pip install --quiet transformers torch
# Expected output: a short install log (or "already satisfied").
# On CPU-only runtimes this pulls the CPU wheel of torch (~200 MB).

## 2. Load GPT-2

`gpt2` is small (117M params, ~500 MB download) and runs on CPU. We use fp32 on CPU, fp16 when a GPU is present.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device == 'cuda' else torch.float32
print('device:', device, '| dtype:', dtype)

model_id = 'gpt2'
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype).to(device)
model.eval()
print('loaded', model_id, '| params:', sum(p.numel() for p in model.parameters()))
# Expected: loaded gpt2 | params: 124439808

## 3. Tokenize: text → token IDs

Tokenization is the front door of inference. GPT-2's vocabulary has 50,257 entries.

In [ ]:
prompt = 'The future of AI is'
ids = tok(prompt, return_tensors='pt')['input_ids'].to(device)
print('prompt:', prompt)
print('token ids:', ids[0].tolist())
print('tokens:', tok.convert_ids_to_tokens(ids[0].tolist()))
# Expected token ids: [464, 3187, 286, 9552, 318]
# Expected tokens: ['The', 'Ġfuture', 'Ġof', 'ĠAI', 'Ġis']  (Ġ = leading space)

## 4. Generate: the decode loop

`model.generate` runs prefill once, then decodes one token at a time (`max_new_tokens=20`). Greedy decoding (`do_sample=False`) keeps the run deterministic.

In [ ]:
import time

n_new = 20
t0 = time.perf_counter()
with torch.inference_mode():
    out = model.generate(ids, max_new_tokens=n_new, do_sample=False,
                         pad_token_id=tok.eos_token_id)
dt = time.perf_counter() - t0

new_ids = out[0][ids.shape[1]:]
print('decoded:', tok.decode(new_ids))
print(f'generated {len(new_ids)} tokens in {dt:.2f} s')
print(f'throughput: {len(new_ids) / dt:.1f} tokens/sec')
# Expected on CPU: ~5-15 tokens/sec. On a T4: ~40-80 tokens/sec.

## 5. KV-cache size: the worked example

Every decoded token leaves a key **and** value vector in every layer:

`kv_bytes = 2 × layers × seq_len × hidden × bytes_per_param`

No torch needed here — pure arithmetic, straight from GPT-2's config.

In [ ]:
layers, hidden = 12, 768          # GPT-2 config
bytes_per_param = 4              # fp32 (use 2 for fp16)

for seq_len in (1024, 2048):
    per_token = 2 * layers * hidden * bytes_per_param
    total = per_token * seq_len
    print(f'seq_len={seq_len}: {per_token:,} bytes/token -> '
          f'{total / 1024**2:.1f} MiB total')
# Expected:
# seq_len=1024: 73,728 bytes/token -> 72.0 MiB total
# seq_len=2048: 73,728 bytes/token -> 144.0 MiB total

## Wrap-up

You just measured the two numbers that govern all of LLM inference: **tokens/sec** (bounded by memory bandwidth in the decode loop) and **KV-cache bytes** (growing linearly with context length).

**Check yourself:**
1. Why did a GPU speed up generation far more than it sped up tokenization?
2. Re-run the KV-cache cell with `bytes_per_param = 2` (fp16). How do the numbers change?
3. What happens to tokens/sec if you raise `n_new` from 20 to 200 — does it stay flat?

Tomorrow (Day 2): inside the attention mechanism — why the KV cache exists at all — and per-layer latency profiling.